In [ ]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

print("Versión de Python:", sys.version.split()[0])

spark = (
    SparkSession.builder
    .appName("VerificacionPySpark")
    .master("local[*]")
    .config("spark.jars.ivy", "C:/SparkLocal/tools/cache/ivy2")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)

# 1. RDD + lambda + map + filter (verifica PYSPARK_PYTHON)
rdd = spark.sparkContext.parallelize([1, 2, 3, 4, 5])
dobles_pares = rdd.map(lambda x: x * 2).filter(lambda x: x > 4).collect()
print("RDD map + lambda:", dobles_pares)

# 2. DataFrame + UDF (verifica worker de Python 3.12)
datos = [("Ana", 9.5), ("Luis", 6.2), ("Sofía", 9.8)]
df = spark.createDataFrame(datos, ["estudiante", "nota"])

@udf(returnType=StringType())
def clasificar_nota(nota):
    return "Sobresaliente" if nota >= 9.0 else "Aprobado"

df_resultado = df.withColumn("estado", clasificar_nota(col("nota")))
df_resultado.show()

# 3. Escritura y lectura Parquet (verifica winutils.exe y hadoop.dll)
ruta_parquet = "C:/SparkLocal/workspace/notebooks/tmp_test_pyspark.parquet"
df_resultado.write.mode("overwrite").parquet(ruta_parquet)
spark.read.parquet(ruta_parquet).show()

spark.stop()